# CoopGCN — Expected Results Preview Notebook

**Purpose:** Render all publication figures and LaTeX tables from expected-results CSVs  
*before* committing to the full training run.  
Replace  with real training outputs → re-run → everything regenerates.

**Datasets:** ML-1M · Amazon-Book  
**Models:** LightGCN · LightGCN++ · HCCF · DyHuCoG · **CoopGCN (Ours)**  
**Metrics @20:** NDCG · Recall · TR (Tail Recall) · Coverage · Gini  

---
| File | Contents |
|---|---|
|  | 5 models × 2 datasets — published + projected numbers |
|  | G1/G2/G3/CL component ablation on ML-1M |
|  | NDCG@20 under 0/5/10/20% injected edge noise |

> ⚠️ CoopGCN values are projections (dim=64, 1000 epochs, patience=50). Replace after training.


In [ ]:

import os, sys, numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg" if not hasattr(matplotlib, "_get_backend_or_none") else "inline")
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from IPython.display import display

matplotlib.rcParams.update({
    "font.family":"DejaVu Sans","axes.spines.top":False,"axes.spines.right":False,
    "axes.grid":True,"grid.alpha":0.3,"grid.linestyle":"--","font.size":11,
})

NB_DIR = os.getcwd()
ROOT   = NB_DIR if os.path.exists("data") else os.path.join(NB_DIR, "..")
DATA   = os.path.join(ROOT, "data")
FIG    = os.path.join(NB_DIR, "figures")
TEX    = os.path.join(NB_DIR, "tables")
os.makedirs(FIG, exist_ok=True); os.makedirs(TEX, exist_ok=True)

COLORS  = {"LightGCN":"#8c8c8c","LightGCN++":"#3aaa6e","HCCF":"#b55cc0",
           "DyHuCoG":"#e07b39","CoopGCN (Ours)":"#1a6faf"}
MARKERS = {"LightGCN":"o","LightGCN++":"s","HCCF":"^","DyHuCoG":"D","CoopGCN (Ours)":"*"}
ORDER   = ["LightGCN","LightGCN++","HCCF","DyHuCoG","CoopGCN (Ours)"]
DS      = ["ML-1M","Amazon-Book"]

df = pd.read_csv(f"{DATA}/expected_results.csv")
da = pd.read_csv(f"{DATA}/expected_ablation.csv")
dn = pd.read_csv(f"{DATA}/expected_noise.csv")
print("✅ Data loaded.")
display(df.pivot_table(index="model",columns="dataset",values=["ndcg_20","recall_20","tr_20","coverage_20"]).round(4))


## Figure 1 — NDCG@20 & Recall@20

In [ ]:

    fig, axes = plt.subplots(1,2,figsize=(14,5.5))
    for ax,(metric,ylabel) in zip(axes,[("ndcg_20","NDCG@20"),("recall_20","Recall@20")]):
        x=np.arange(len(DS)); n=len(ORDER); w=0.15
        off=np.linspace(-(n-1)/2*w,(n-1)/2*w,n)
        for i,m in enumerate(ORDER):
            vals=[df[(df.dataset==ds)&(df.model==m)][metric].values[0] for ds in DS]
            bars=ax.bar(x+off[i],vals,w,color=COLORS[m],label=m,
                        alpha=0.88 if m!="CoopGCN (Ours)" else 1.0,
                        edgecolor="black" if m=="CoopGCN (Ours)" else "none",linewidth=1.2)
            for bar,v in zip(bars,vals):
                ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.003,f"{v:.4f}",
                        ha="center",va="bottom",fontsize=6.5,color=COLORS[m],
                        fontweight="bold" if m=="CoopGCN (Ours)" else "normal")
        ax.set_xticks(x); ax.set_xticklabels(DS,fontsize=12)
        ax.set_ylabel(ylabel,fontsize=12); ax.set_title(ylabel,fontsize=13,fontweight="bold")
        ax.set_ylim(0,ax.get_ylim()[1]*1.18)
    axes[0].legend(fontsize=9.5,loc="upper right")
    fig.suptitle("Overall Recommendation Accuracy — Graph CF Family
(ML-1M · Amazon-Book · Full-catalog ranking @20)",fontsize=12,fontweight="bold",y=1.02)
    plt.tight_layout(); fig.savefig(f"{FIG}/fig1_main_performance.png",dpi=200,bbox_inches="tight"); plt.show()


## Figure 2 — TR@20 & Catalog Coverage@20

In [ ]:

fig,axes=plt.subplots(1,2,figsize=(14,5.5))
for ax,(metric,ylabel,fmt) in zip(axes,[("tr_20","Tail Recall TR@20",".4f"),("coverage_20","Catalog Coverage@20",".3f")]):
    x=np.arange(len(DS)); off=np.linspace(-(len(ORDER)-1)/2*0.15,(len(ORDER)-1)/2*0.15,len(ORDER))
    for i,m in enumerate(ORDER):
        vals=[df[(df.dataset==ds)&(df.model==m)][metric].values[0] for ds in DS]
        bars=ax.bar(x+off[i],vals,0.15,color=COLORS[m],label=m,
                    alpha=0.88 if m!="CoopGCN (Ours)" else 1.0,
                    edgecolor="black" if m=="CoopGCN (Ours)" else "none",linewidth=1.2)
        for bar,v in zip(bars,vals):
            if v>0.0001:
                ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+df[metric].max()*0.015,
                        f"{v:{fmt}}",ha="center",va="bottom",fontsize=6.5,color=COLORS[m],
                        fontweight="bold" if m=="CoopGCN (Ours)" else "normal")
    ax.set_xticks(x); ax.set_xticklabels(DS,fontsize=12)
    ax.set_ylabel(ylabel,fontsize=11); ax.set_title(ylabel,fontsize=13,fontweight="bold")
    ax.set_ylim(0,max(df[metric].max()*1.22,0.001))
    if metric=="tr_20": ax.text(0.02,0.97,"LightGCN/HCCF/DyHuCoG = 0.0000 on both datasets",transform=ax.transAxes,fontsize=8,color="#888",va="top",style="italic")
axes[0].legend(fontsize=9.5,loc="upper right")
fig.suptitle("Popularity-Bias Mitigation — CoopGCN #1 on TR@20 & Coverage",fontsize=12,fontweight="bold",y=1.02)
plt.tight_layout(); fig.savefig(f"{FIG}/fig2_tail_coverage.png",dpi=200,bbox_inches="tight"); plt.show()


## Figure 3 — Component Ablation Study (ML-1M)

In [ ]:

abl_labels=da["variant"].tolist()
abl_colors=["#8c8c8c" if "LightGCN" in v else "#e07b39" if "DyHuCoG" in v else "#aaaaaa" if "w/o" in v else "#1a6faf" for v in abl_labels]
fig,axes=plt.subplots(1,3,figsize=(15,5.5))
for ax,(metric,ylabel,fmt) in zip(axes,[("ndcg_20","NDCG@20",".4f"),("tr_20","Tail Recall TR@20",".4f"),("coverage_20","Catalog Coverage@20",".3f")]):
    vals=da[metric].values; xa=np.arange(len(abl_labels))
    bars=ax.bar(xa,vals,color=abl_colors,alpha=0.88,edgecolor="black",linewidth=0.6)
    bars[-1].set_edgecolor("#1a6faf"); bars[-1].set_linewidth(2.5)
    for bar,v in zip(bars,vals): ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+max(vals)*0.012,f"{v:{fmt}}",ha="center",va="bottom",fontsize=8)
    ax.set_xticks(xa); ax.set_xticklabels(abl_labels,fontsize=7.5,rotation=20,ha="right")
    ax.set_ylabel(ylabel,fontsize=11); ax.set_title(ylabel,fontsize=12,fontweight="bold"); ax.set_ylim(0,max(vals)*1.22)
fig.suptitle("Component Ablation Study (ML-1M) — Each G1/G2/G3/CL component contributes",fontsize=12,fontweight="bold",y=1.02)
plt.tight_layout(); fig.savefig(f"{FIG}/fig3_ablation.png",dpi=200,bbox_inches="tight"); plt.show()


## Figure 4 — Adversarial Noise Immunity (ML-1M)

In [ ]:

    nms=["LightGCN","LightGCN++","HCCF","DyHuCoG","CoopGCN (Ours)"]
    nr=dn["noise_ratio"].values*100
    fig,ax=plt.subplots(figsize=(9,5.5))
    for m in nms:
        vals=dn[m].values; lw=2.8 if m=="CoopGCN (Ours)" else 1.6
        ax.plot(nr,vals,marker=MARKERS[m],linewidth=lw,linestyle="-" if m=="CoopGCN (Ours)" else "--",
                color=COLORS[m],label=m,markersize=9 if lw>2 else 6,zorder=5 if lw>2 else 3)
        drop=(vals[-1]-vals[0])/vals[0]*100
        ax.text(nr[-1]+0.3,vals[-1],f"{drop:+.1f}%",va="center",fontsize=8.5,color=COLORS[m],fontweight="bold" if m=="CoopGCN (Ours)" else "normal")
    ax.set_xlabel("Edge Noise Ratio (%)",fontsize=12); ax.set_ylabel("NDCG@20",fontsize=12)
    ax.set_title("Adversarial Robustness — NDCG@20 under injected edge noise (ML-1M)
CoopGCN degrades least — G3 Data-Shapley pruning attenuates noisy edges",fontsize=12,fontweight="bold")
    ax.set_xticks(nr); ax.set_xticklabels([f"{r:.0f}%" for r in nr]); ax.legend(fontsize=10); ax.set_xlim(-1,nr[-1]+3)
    plt.tight_layout(); fig.savefig(f"{FIG}/fig4_noise_immunity.png",dpi=200,bbox_inches="tight"); plt.show()


## Figure 5 — Gain Heatmap vs DyHuCoG

In [ ]:

    metrics_hm=["ndcg_20","recall_20","tr_20","coverage_20","gini"]
    mlabels=["NDCG@20","Recall@20","TR@20","Coverage@20","Gini↓"]
    gcap=[]; greal=[]
    for ds in DS:
        coop=df[(df.dataset==ds)&(df.model=="CoopGCN (Ours)")].iloc[0]
        ref=df[(df.dataset==ds)&(df.model=="DyHuCoG")].iloc[0]
        rc=[]; rr=[]
        for m in metrics_hm:
            if m=="gini": g=(ref[m]-coop[m])/ref[m]*100; inf=False
            elif ref[m]<1e-6: g=60.0; inf=(coop[m]>0)
            else: g=(coop[m]-ref[m])/ref[m]*100; inf=False
            rc.append(min(max(g,-15),60)); rr.append((g,inf))
        gcap.append(rc); greal.append(rr)
    ga=np.array(gcap)
    norm=TwoSlopeNorm(vmin=-15,vcenter=0,vmax=60)
    fig,ax=plt.subplots(figsize=(10,3.8))
    im=ax.imshow(ga,cmap="RdYlGn",aspect="auto",norm=norm)
    ax.set_xticks(range(len(mlabels))); ax.set_xticklabels(mlabels,fontsize=12,fontweight="bold")
    ax.set_yticks(range(len(DS))); ax.set_yticklabels(DS,fontsize=13,fontweight="bold")
    for i,row in enumerate(greal):
        for j,(v,inf) in enumerate(row):
            txt="+∞" if inf else (f"+{v:.1f}%" if v>=0 else f"{v:.1f}%")
            tc="white" if abs(ga[i,j])>35 else "black"
            ax.text(j,i,txt,ha="center",va="center",fontsize=13,fontweight="bold",color=tc)
    for i in range(len(DS)):
        for j in range(len(metrics_hm)): ax.add_patch(plt.Rectangle((j-.5,i-.5),1,1,fill=False,edgecolor="white",linewidth=2))
    fig.colorbar(im,ax=ax,fraction=0.025,pad=0.02).set_label("Gain over DyHuCoG (%)",fontsize=10)
    ax.set_title("CoopGCN vs. DyHuCoG — Relative Improvement (%)
Green=better · +∞ where DyHuCoG=0.0000 · All green = win on every metric",fontsize=11,fontweight="bold",pad=10)
    plt.tight_layout(); fig.savefig(f"{FIG}/fig5_gain_heatmap.png",dpi=200,bbox_inches="tight"); plt.show()


## Figure 6 — Holistic Radar (ML-1M)

In [ ]:

    N=5; angles=np.linspace(0,2*np.pi,N,endpoint=False).tolist(); angles+=angles[:1]
    rlabels=["NDCG@20","Recall@20","TR@20
(×50)","Coverage@20","Equity
(1-Gini)"]
    fig,ax=plt.subplots(figsize=(7,7),subplot_kw=dict(polar=True))
    for m in ORDER:
        row=df[(df.dataset=="ML-1M")&(df.model==m)].iloc[0]
        vals=[row["ndcg_20"],row["recall_20"],row["tr_20"]*50,row["coverage_20"],1-row["gini"]]; vals+=vals[:1]
        lw=2.8 if m=="CoopGCN (Ours)" else 1.5
        ax.plot(angles,vals,linewidth=lw,color=COLORS[m],label=m,linestyle="-" if m=="CoopGCN (Ours)" else "--")
        ax.fill(angles,vals,alpha=0.08 if m!="CoopGCN (Ours)" else 0.18,color=COLORS[m])
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(rlabels,fontsize=11)
    ax.set_title("Holistic Performance Radar — ML-1M
CoopGCN dominates TR@20, Coverage, Equity",fontsize=12,fontweight="bold",pad=20)
    ax.legend(loc="upper right",bbox_to_anchor=(1.38,1.15),fontsize=9.5)
    plt.tight_layout(); fig.savefig(f"{FIG}/fig6_radar.png",dpi=200,bbox_inches="tight"); plt.show()


## LaTeX Tables — generate all 4

In [ ]:

    def bold(v,best,lo=False):
        s=f"{v:.4f}"
        return f"\textbf{{{s}}}" if v==best else s

    # Table 1
    lines=[r"egin{table*}[t]",r"\centering",
        r"\caption{Overall recommendation performance (ML-1M + Amazon-Book, full-catalog @20). Best per column in 	extbf{bold}. $\dagger$=projected.}",
        r"\label{tab:overall}",r"esizebox{	extwidth}{!}{",
        r"egin{tabular}{ll ccccc}",r"	oprule",
        r"	extbf{Dataset} & 	extbf{Model} & 	extbf{NDCG@20} & 	extbf{Recall@20} & 	extbf{TR@20} & 	extbf{Cov@20} & 	extbf{Gini$\downarrow$} \\",r"\midrule"]
    for di,ds in enumerate(DS):
        sub=df[df.dataset==ds].set_index("model")
        bn,br,bt,bc,bg=sub.ndcg_20.max(),sub.recall_20.max(),sub.tr_20.max(),sub.coverage_20.max(),sub.gini.min()
        for ji,m in enumerate(["LightGCN","LightGCN++","HCCF","DyHuCoG","CoopGCN (Ours)"]):
            row=sub.loc[m]; dsl=ds if ji==0 else ""; dag=r"$^{\dagger}$" if "Ours" in m else ""
            lines.append(f"{dsl} & {m}{dag} & {bold(row.ndcg_20,bn)} & {bold(row.recall_20,br)} & {bold(row.tr_20,bt)} & {bold(row.coverage_20,bc)} & {bold(row.gini,bg)} \\")
        if di<len(DS)-1: lines.append(r"\midrule")
    lines+=[r"ottomrule",r"\end{tabular}}",r"\end{table*}"]
    tex1="
".join(lines)
    with open(f"{TEX}/tab1_overall.tex","w") as f: f.write(tex1)

    # Table 2
    lines=[r"egin{table}[t]\centering",
        r"\caption{Component ablation on ML-1M.}",
        r"\label{tab:ablation}",r"egin{tabular}{l cccc}	oprule",
        r"	extbf{Variant} & 	extbf{NDCG@20} & 	extbf{TR@20} & 	extbf{Cov@20} & 	extbf{Gini$\downarrow$} \\",r"\midrule"]
    bn,bt,bc,bg=da.ndcg_20.max(),da.tr_20.max(),da.coverage_20.max(),da.gini.min()
    for _,row in da.iterrows():
        if "Full" in row["variant"]: lines.append(r"\midrule")
        lines.append(f"{row['variant']} & {bold(row.ndcg_20,bn)} & {bold(row.tr_20,bt)} & {bold(row.coverage_20,bc)} & {bold(row.gini,bg)} \\")
    lines+=[r"ottomrule",r"\end{tabular}",r"\end{table}"]
    with open(f"{TEX}/tab2_ablation.tex","w") as f: f.write("
".join(lines))

    # Table 3
    nms=["LightGCN","LightGCN++","HCCF","DyHuCoG","CoopGCN (Ours)"]
    lines=[r"egin{table}[t]\centering",
        r"\caption{NDCG@20 under adversarial edge noise (ML-1M). Best per row in 	extbf{bold}.}",
        r"\label{tab:noise}",f"\begin{{tabular}}{{l {{\small c}}{{'\small c'}}{{'\small c'}}{{'\small c'}}{{'\small c'}}}}\toprule"]
    lines=[r"egin{table}[t]\centering",
        r"\caption{NDCG@20 under adversarial edge noise injection (ML-1M). $\Delta_{20\%}$=relative drop at 20\% noise.}",
        r"\label{tab:noise}",r"egin{tabular}{l ccccc}	oprule",
        "\textbf{Noise} & "+" & ".join(f"\textbf{{{m}}}" for m in nms)+r" \\",r"\midrule"]
    for _,row in dn.iterrows():
        r=f"{int(row['noise_ratio']*100)}\%"; vals=[row[m] for m in nms]; best=max(vals)
        cells=[f"\textbf{{{v:.4f}}}" if v==best else f"{v:.4f}" for v in vals]
        lines.append(f"{r} & {' & '.join(cells)} \\")
    lines.append(r"\midrule")
    deltas=[(dn[dn.noise_ratio==0.2][m].values[0]-dn[dn.noise_ratio==0.0][m].values[0])/dn[dn.noise_ratio==0.0][m].values[0]*100 for m in nms]
    bd=max(deltas)
    lines.append("$\Delta_{20\%}$ & "+" & ".join(f"\textbf{{{d:+.1f}\%}}" if d==bd else f"{d:+.1f}\%" for d in deltas)+" \\")
    lines+=[r"ottomrule",r"\end{tabular}",r"\end{table}"]
    with open(f"{TEX}/tab3_noise.tex","w") as f: f.write("
".join(lines))

    # Table 4
    lines=[r"egin{table}[t]\centering",
        r"\caption{CoopGCN relative gain over DyHuCoG. $+\infty$ where DyHuCoG=0.0000.}",
        r"\label{tab:gain}",r"egin{tabular}{l ccccc}	oprule",
        r"	extbf{Dataset} & $\Delta$	extbf{NDCG} & $\Delta$	extbf{Recall} & $\Delta$	extbf{TR@20} & $\Delta$	extbf{Cov} & $\Delta$	extbf{Gini$\downarrow$} \\",r"\midrule"]
    for ds in DS:
        coop=df[(df.dataset==ds)&(df.model=="CoopGCN (Ours)")].iloc[0]
        ref=df[(df.dataset==ds)&(df.model=="DyHuCoG")].iloc[0]
        cells=[]
        for m,lo in [("ndcg_20",False),("recall_20",False),("tr_20",False),("coverage_20",False),("gini",True)]:
            if lo: cells.append(f"\textbf{{+{(ref[m]-coop[m])/ref[m]*100:.1f}\%}}")
            elif ref[m]<1e-6: cells.append(r"$+\infty$")
            else:
                g=(coop[m]-ref[m])/ref[m]*100
                cells.append(f"\textbf{{+{g:.1f}\%}}" if g>0 else f"{g:.1f}\%")
        lines.append(f"{ds} & {' & '.join(cells)} \\")
    lines+=[r"ottomrule",r"\end{tabular}",r"\end{table}"]
    with open(f"{TEX}/tab4_gain.tex","w") as f: f.write("
".join(lines))

    print("✅ All 4 LaTeX tables saved to", TEX)
    for t in ["tab1_overall","tab2_ablation","tab3_noise","tab4_gain"]:
        print(f"  {t}.tex")


## Summary

In [ ]:

import glob
figs=sorted(glob.glob(f"{FIG}/fig*.png"))
tabs=sorted(glob.glob(f"{TEX}/tab*.tex"))
print(f"Figures generated: {len(figs)}")
for f in figs: print(f"  {os.path.basename(f)}")
print(f"LaTeX tables:      {len(tabs)}")
for t in tabs: print(f"  {os.path.basename(t)}")
print()
print("Next steps:")
print("  1. Run training: ML-1M first (~4 days on M4 Pro)")
print("  2. Replace data/expected_results.csv with real results")
print("  3. Re-run this notebook → all figures + tables update automatically")
